# XLM-R arm — Colab / Kaggle

Fine-tunes `xlm-roberta-base` for the 7 HumSet labels, 5-fold CV on the 383
hand-labelled articles. Same protocol as the local TF-IDF arm.

**Compare the result against macro-F1\* 0.727**, not 0.755. XLM-R reads only the
first 512 tokens; 0.727 is TF-IDF on that same truncated text. 0.755 is TF-IDF on
full articles and is the wrong baseline for this arm.

| run | macro-F1\* | exact |
|---|---|---|
| TF-IDF full text | 0.755 | 0.569 |
| **TF-IDF matched (truncated)** | **0.727** | 0.567 |
| XLM-R, local M2, 1 seed | 0.640 | 0.389 |

### ⚠️ Before you upload: the article text is copyrighted

The project's own rule is that article text stays private and only URLs, labels,
metadata and scripts are released. `humset_task.jsonl` contains full article
bodies.

Uploading it to Colab or Kaggle is **not** publishing it, but it does send it to
a third party. Keep it private:

- **Colab** — upload to your own Google Drive and mount it, or use the session
  file picker. Do not put it in a public notebook or a public GitHub repo.
- **Kaggle** — create the dataset as **Private**. Kaggle datasets default to
  private but confirm it before uploading.

If you would rather not upload bodies at all, the alternative is to ship only
URLs + labels and re-fetch text in the notebook — but that needs the crawler and
re-hits the outlets, so uploading a private copy is the lesser evil.


## 1. Check the GPU

Colab: *Runtime → Change runtime type → T4 GPU*.
Kaggle: *Settings → Accelerator → GPU T4 x2*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, platform
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("bf16 supported:", torch.cuda.is_bf16_supported())

## 2. Upload the project files

You need **8 files**. Zip them locally first:

```bash
cd "/path/to/MyHumNews"
zip -r xlmr_bundle.zip \
    classifier/__init__.py classifier/humset.py \
    classifier/train_humset.py classifier/train_xlmr.py \
    classifier/data/humset_gold.jsonl classifier/data/humset_task.jsonl \
    classifier/data/humset_round2.jsonl classifier/data/humset_thin_task.jsonl
```

Then run the cell below and pick `xlmr_bundle.zip`.

On **Kaggle**, instead add it as a private Dataset and set
`BUNDLE = "/kaggle/input/<your-dataset>/xlmr_bundle.zip"`.

In [ ]:
import os, glob, zipfile, pathlib

ZIP = None   # Kaggle: set to "/kaggle/input/<dataset>/xlmr_bundle.zip"

if ZIP is None:
    from google.colab import drive
    drive.mount("/content/drive")          # harmless if already mounted
    hits = glob.glob("/content/drive/MyDrive/**/xlmr_bundle.zip", recursive=True)
    if not hits:
        raise SystemExit("xlmr_bundle.zip not found under /content/drive/MyDrive/ "
                         "-- upload it to drive.google.com, then re-run this cell.")
    ZIP = hits[0]
print("bundle:", ZIP)

WORK = "/content/work" if os.path.isdir("/content") else "./work"
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(WORK)
os.chdir(WORK)                             # os.chdir, NOT !cd -- !cd runs in a
                                           # subshell and the change is lost.
print("cwd:", os.getcwd())
need = ["classifier/humset.py", "classifier/train_xlmr.py",
        "classifier/data/humset_gold.jsonl", "classifier/data/humset_task.jsonl"]
missing = [f for f in need if not os.path.exists(f)]
if missing:
    raise SystemExit(f"bundle is incomplete, missing: {missing}")
for f in sorted(pathlib.Path(".").rglob("*")):
    if f.is_file():
        print(f"  {f}  {f.stat().st_size/1024:,.0f} KB")
print("\nall required files present")

## 3. Install transformers

Colab ships torch already; only `transformers` and `scikit-learn` may need pinning.

In [ ]:
!pip -q install "transformers>=4.44" "scikit-learn>=1.4"
import transformers, sklearn
print("transformers", transformers.__version__, "| sklearn", sklearn.__version__)

## 4. Keep the checkpoint on Drive (Colab only, recommended)

Colab disconnects. The trainer writes each fold's scores as soon as it finishes
and skips completed folds on a re-run, so pointing the checkpoint at Drive means
a disconnect costs you one fold, not five.

Skip this cell on Kaggle (its working dir persists within a session).

In [ ]:
CKPT = "classifier/data/xlmr_ckpt_aug.json"
DUMP = "classifier/data/humset_xlmr_colab_aug.jsonl"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/myhumnews", exist_ok=True)
    # BOTH must live on Drive. A checkpoint that survives a disconnect is no
    # use if the predictions it produced died with the runtime.
    CKPT = "/content/drive/MyDrive/myhumnews/xlmr_ckpt_aug.json"
    DUMP = "/content/drive/MyDrive/myhumnews/humset_xlmr_colab_aug.jsonl"
except Exception as e:
    print("no Drive -- checkpoint and dump are on the runtime disk and will be")
    print("LOST if this session disconnects. Fix the mount before training:", e)
print("checkpoint ->", CKPT)
print("dump       ->", DUMP)


# The _aug filenames keep this run clear of the gold-only checkpoints already
# in Drive (xlmr_ckpt_s2026101{5,6,7}.json) that produced the 0.621 result.
# If you set AUG = "" below, change these names back or you will mix the two.

# Round 2's 180 training articles, appended to every CV fold's TRAINING split
# only -- the test folds stay pure gold, so the score still describes the
# population gold was sampled from. Set AUG = "" to run the gold-only arm.
# NOTE: checkpoints record how many augment rows were used, so an augmented run
# will NOT resume from a gold-only checkpoint (or vice versa). That is deliberate.
AUG = ("--augment classifier/data/humset_round2.jsonl "
       "--augment-task classifier/data/humset_thin_task.jsonl")
print("augment    ->", AUG or "(none -- gold only)")

## 5. Probe the tokenizer first

Confirms the truncation figures the comparison depends on: ~60% of articles cut,
~30% of tokens discarded. Costs nothing, no training.

In [ ]:
!python -m classifier.train_xlmr --probe

## 6. Train

`--no-grad-ckpt` is on because a T4's 16 GB is dedicated VRAM, unlike the
laptop's shared 17 GB — faster without it. Mixed precision is automatic on CUDA
(bf16 where supported, else fp16 + loss scaling).

Expect roughly **10–20 minutes** on a T4, against ~1.6 h on the M2.

If the session drops, **re-run this same cell** — it resumes from the checkpoint.

In [ ]:
!python -m classifier.train_xlmr \
    --epochs 4 --batch 16 --lr 2e-5 \
    --no-grad-ckpt $AUG \
    --ckpt "$CKPT" \
    --dump "$DUMP"


## 7. Multiple seeds

A single fine-tuning run on 306 examples per fold is noisy. The local TF-IDF arm
needed a paired test across splits to separate a 0.015 effect from 0.055 fold
noise, and fine-tuning is noisier still.

The 0.640-vs-0.727 gap is 0.087, which is probably real — but three seeds settle
it. On a T4 that is under an hour.

In [ ]:
!python -m classifier.train_xlmr \
    --epochs 4 --batch 16 --lr 2e-5 --seeds 3 \
    --no-grad-ckpt $AUG \
    --ckpt "$CKPT" 

## 8. Download the predictions

Bring the per-article predictions back so they can be scored and compared locally.

In [ ]:
import glob, os

outs = sorted(set(
    glob.glob("classifier/data/humset_xlmr_colab*.jsonl")
    + glob.glob(os.path.join(os.path.dirname(DUMP) or ".",
                             "humset_xlmr_colab*.jsonl"))
))
print(outs)

if DUMP.startswith("/content/drive/"):
    print("\nalready on your Drive -- these survive a disconnect, the download")
    print("below is only a convenience.")

try:
    from google.colab import files
    for o in outs:
        files.download(o)
except ImportError:
    print("On Kaggle: these are in /kaggle/working and downloadable from the Output tab.")


## What to do with the result

Score it against the gold set exactly as the local arms are scored:

```bash
python -m classifier.score_multilabel \
    --gold classifier/data/humset_gold.jsonl \
    --pred classifier/data/humset_xlmr_colab.jsonl
```

**Report macro-F1 over the 5 labels with support ≥ 20**, and report
`humanitarian_access` (16) and `information_communication` (6) as thin support
with counts, never as F1 scores. A per-label F1 on 6 positives swings 0.17 per
article.

One result worth checking for specifically: on the local run XLM-R was the
**only** model to score non-zero on `humanitarian_access` (0.139 against TF-IDF's
0.000). If that holds across seeds it is real evidence for the shared-head
argument — a transformer with one head over 7 labels borrows signal across them,
which one-vs-rest TF-IDF cannot do.
